# 🗳️ Notebook 1: Quorum Reads & Writes (W + R > N)

**Setup:** N replicas. A write is acknowledged once W replicas confirm it. A read polls R replicas and returns the freshest.

If **W + R > N** then any read overlaps with the most recent write — you're guaranteed to see it.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/quorum
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List
import random

@dataclass
class Replica:
    name: str
    up: bool = True
    data: Dict[str, Tuple[str,int]] = field(default_factory=dict)
    def write(self, k, v, ts):
        if not self.up: return False
        cur = self.data.get(k)
        if cur is None or ts > cur[1]: self.data[k] = (v, ts)
        return True
    def read(self, k):
        if not self.up: return None
        return self.data.get(k)

class Cluster:
    def __init__(self, n=5):
        self.replicas = [Replica(f'r{i}') for i in range(n)]
        self.n = n
        self._ts = 0

    def write(self, k, v, W):
        self._ts += 1
        ts = self._ts
        acks = 0
        for r in self.replicas:
            if r.write(k, v, ts):
                acks += 1
                if acks >= W:
                    return True, ts, acks
        return False, ts, acks

    def read(self, k, R):
        responses = []
        for r in self.replicas:
            val = r.read(k)
            if val is not None:
                responses.append((r.name, val))
                if len(responses) >= R:
                    break
        if not responses: return None
        return max(responses, key=lambda x: x[1][1])  # freshest by timestamp


## ✅ Strong consistency: W=3, R=3, N=5  → W+R=6 > N

In [ ]:
c = Cluster(n=5)
ok, ts, acks = c.write('x', 'first',  W=3); print('write1:', ok, ts, 'acks=',acks)
ok, ts, acks = c.write('x', 'second', W=3); print('write2:', ok, ts, 'acks=',acks)
print('read W+R>N:', c.read('x', R=3))


## ⚠️ Eventual consistency: W=1, R=1, N=5 → W+R=2 ≤ N

In [ ]:
c = Cluster(n=5)
# Force write to land on only the first replica by killing the rest temporarily
for r in c.replicas[1:]: r.up = False
c.write('x', 'fresh', W=1)
for r in c.replicas: r.up = True

# Read with R=1 might hit any replica — including a stale one
random.seed(1)
for _ in range(5):
    random.shuffle(c.replicas)
    print('read:', c.read('x', R=1))


## 📊 Tradeoff cheatsheet (N=5)

| W | R | Property | Best for |
|---|---|---|---|
| 5 | 1 | strong, fast reads | read-heavy, can tolerate write outages |
| 1 | 5 | strong, fast writes | write-heavy logs |
| 3 | 3 | strong, balanced | most OLTP workloads |
| 1 | 1 | eventual | maximum availability, can tolerate stale reads |

Picking W and R is the **dial** between consistency, availability, and latency in DynamoDB / Cassandra / Riak.